# 02 — SQL Analysis

This notebook uses SQL queries to answer three business questions about the Superstore dataset.  
All queries run against the SQLite database created in `01_setup_database.ipynb`.

**Business Questions:**
1. Which region has the lowest profit margin?
2. Why does that region underperform — is discounting the cause?
3. Which specific sub-categories are losing money in that region?

The goal is to move from a broad observation ("Central is unprofitable") to a specific, actionable root cause ("certain sub-categories are being over-discounted").

In [78]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("../data/superstore.db")
print("Connected to superstore.db")

Connected to superstore.db


## Step 1 — Explore Table Structure

Before writing business queries, we inspect the schema to confirm column names and data types.  
This prevents errors from typos or wrong type assumptions (e.g. treating a text column as numeric).

In [79]:
col_query = "PRAGMA table_info(orders)"
columns = pd.read_sql(col_query, conn)
print(columns[['name', 'type']])

             name     type
0          Row ID  INTEGER
1        Order ID     TEXT
2      Order Date     TEXT
3       Ship Date     TEXT
4       Ship Mode     TEXT
5     Customer ID     TEXT
6   Customer Name     TEXT
7         Segment     TEXT
8         Country     TEXT
9            City     TEXT
10          State     TEXT
11    Postal Code  INTEGER
12         Region     TEXT
13     Product ID     TEXT
14       Category     TEXT
15   Sub-Category     TEXT
16   Product Name     TEXT
17          Sales     REAL
18       Quantity  INTEGER
19       Discount     REAL
20         Profit     REAL


In [80]:
# Or simply read the first 2 rows to see column names visually
preview = pd.read_sql("SELECT * FROM orders LIMIT 2", conn)
print(preview.columns.tolist())

['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


**Column Reference**

| Column | Type | Description |
|--------|------|-------------|
| Row ID | INTEGER | Sequential row number, no business meaning |
| Order ID | TEXT | Order identifier — one order can have multiple rows (one per product) |
| Order Date | TEXT | Date the order was placed |
| Ship Date | TEXT | Date the order was shipped — subtract from Order Date to get lead time |
| Ship Mode | TEXT | Shipping method: First Class, Second Class, Standard Class, Same Day |
| Customer ID | TEXT | Unique customer identifier |
| Customer Name | TEXT | Customer's full name |
| Segment | TEXT | Customer type: Consumer, Corporate, Home Office |
| Country | TEXT | Country — all rows are United States in this dataset |
| City | TEXT | City |
| State | TEXT | US state |
| Postal Code | INTEGER | ZIP code |
| Region | TEXT | Sales region: West, East, Central, South — key dimension in this analysis |
| Product ID | TEXT | Unique product identifier |
| Category | TEXT | Product category: Furniture, Office Supplies, Technology |
| Sub-Category | TEXT | Product sub-category (17 types, e.g. Chairs, Binders, Phones) |
| Product Name | TEXT | Full product name |
| Sales | REAL | Revenue after discount — the actual amount the customer paid |
| Quantity | INTEGER | Number of units purchased |
| Discount | REAL | Discount rate from 0 to 1 (e.g. 0.3 = 30% off) |
| Profit | REAL | Profit amount — can be negative (loss) |

## Step 2 — Business Question 1: Which region has the lowest profit margin?

**What is profit margin?**  
Profit margin (`profit_margin_pct`) measures how much profit a region keeps for every dollar of revenue — e.g., a 14.94% margin means $0.15 of profit per $1 of sales. It is a better comparison metric than total profit, because it normalises for sales volume: a region with $1M in sales and $50K profit is less efficient than one with $500K in sales and $50K profit.

**Why `SUM(Profit) / SUM(Sales)` and not `AVG(Profit/Sales)`?**  
The former gives the true aggregate margin; the latter would give each order equal weight regardless of size, which would distort the result for high-volume regions.

In [81]:
#將sale 加種、四捨五入至小數點後兩位，然後將結果設為total_sales
#將profit 加種、四捨五入至小數點後兩位，然後將結果設為total_profit
query = """
    SELECT 
        Region,
        ROUND(SUM(Sales), 2) AS total_sales,
        ROUND(SUM(Profit), 2) AS total_profit,
        ROUND(SUM(Profit) / SUM(Sales) * 100, 2) AS profit_margin_pct
    FROM orders
    GROUP BY Region
    ORDER BY total_profit DESC
"""

region_df = pd.read_sql(query, conn)
region_df

,Region,total_sales,total_profit,profit_margin_pct
0,West,725457.82,108418.45,14.94
1,East,678781.24,91522.78,13.48
2,South,391721.91,46749.43,11.93
3,Central,501239.89,39706.36,7.92


**Conclusion:** Central has the lowest profit margin at **7.92%** — nearly half of West's 14.94%, despite ranking 3rd in total sales ($501K). This makes Central the primary target for further investigation.

## Step 3 — Business Question 2: Why does Central underperform?

Central has the lowest margin (7.92%) despite ranking 3rd in total sales.  
The hypothesis is that excessive discounting is eroding profit.

Before comparing regions, we first need to establish an empirically grounded threshold for "high discount".  
**There is no universal industry standard** — so instead of assuming a number (e.g. 30%), we let the data tell us at which discount level orders start losing money.

### Step 3a — Find the empirical breakeven discount

We bucket all Central orders by discount range and calculate average profit per bucket.  
The discount level where `avg_profit` turns negative is our empirical threshold.

In [82]:
query = """
    SELECT
        CASE
            WHEN Discount = 0        THEN '0%'
            WHEN Discount <= 0.10    THEN '1–10%'
            WHEN Discount <= 0.20    THEN '11–20%'
            WHEN Discount <= 0.30    THEN '21–30%'
            WHEN Discount <= 0.40    THEN '31–40%'
            ELSE                          '40%+'
        END AS discount_bucket,
        ROUND(AVG(Profit), 2) AS avg_profit,
        ROUND(AVG(Discount) * 100, 1) AS avg_discount_pct,
        COUNT(*) AS order_count
    FROM orders
    WHERE Region = 'Central'
    GROUP BY discount_bucket
    ORDER BY avg_discount_pct
"""

bucket_df = pd.read_sql(query, conn)
bucket_df

,discount_bucket,avg_profit,avg_discount_pct,order_count
0,0%,91.94,0.0,828
1,1–10%,106.56,10.0,18
2,11–20%,16.85,20.0,834
3,21–30%,-44.50,30.0,147
4,31–40%,-126.45,34.6,40
5,40%+,-89.46,72.8,456


**Finding:** Average profit turns negative in the **21–30% bucket** (avg_profit = −$44.50).  
Orders with no discount or low discount (≤20%) are consistently profitable. Orders discounted above 20% begin losing money on average.  
→ We set the "high discount" threshold at **20%** for the regional comparison below.

### Step 3b — Compare discount behaviour across regions

The bucket analysis shows that orders discounted above **20%** become unprofitable on average.  
We now apply this threshold across all four regions to measure how many orders in each region exceed it.

In [83]:
# Threshold set at 0.2 (20%) — empirically derived in Step 3a:
# avg_profit turns negative in the 21–30% discount bucket, so >20% is the breakeven point.
query = """
    SELECT
        region,
        ROUND(AVG(discount), 2) AS avg_discount,
        ROUND(MAX(discount), 2) AS max_discount,
        COUNT(CASE WHEN discount > 0.2 THEN 1 END) AS high_discount_orders,
        COUNT(*) AS total_orders
    FROM orders
    GROUP BY region
    ORDER BY avg_discount DESC
"""
discount_df = pd.read_sql(query, conn)
discount_df

,Region,avg_discount,max_discount,high_discount_orders,total_orders
0,Central,0.24,0.8,643,2323
1,South,0.15,0.7,171,1620
2,East,0.15,0.7,461,2848
3,West,0.11,0.7,118,3203


### Step 3c — Does high discount rate correlate with lower profit margin?

We shift the unit of analysis from **Region to State** (n ≈ 49) to get sufficient statistical power for Pearson correlation.

For each state we calculate two metrics:
- `profit_margin_pct` — SUM(Profit) / SUM(Sales)
- `high_discount_rate` — share of orders with Discount > 20% (our empirical threshold from Step 3a)

A strong negative correlation (r close to −1, p < 0.05) would confirm that discounting is the structural driver of low margins — not a Central-specific anomaly.

In [84]:
from scipy import stats

query = """
    SELECT
        State,
        ROUND(SUM(Profit) / SUM(Sales) * 100, 2) AS profit_margin_pct,
        ROUND(
            CAST(COUNT(CASE WHEN Discount > 0.2 THEN 1 END) AS REAL) / COUNT(*),
        4) AS high_discount_rate,
        COUNT(*) AS total_orders
    FROM orders
    GROUP BY State
    HAVING total_orders >= 10
    ORDER BY profit_margin_pct
"""

state_df = pd.read_sql(query, conn)

r, p_value = stats.pearsonr(state_df["high_discount_rate"], state_df["profit_margin_pct"])

print(f"n = {len(state_df)} states")
print(f"Pearson r = {r:.3f}")
print(f"p-value   = {p_value:.4f}")
print()
if p_value < 0.05:
    print("Result: statistically significant (p < 0.05)")
else:
    print("Result: not statistically significant (p ≥ 0.05)")

state_df.head(10)  # bottom 10 states by profit margin

n = 45 states
Pearson r = -0.905
p-value   = 0.0000

Result: statistically significant (p < 0.05)


,State,profit_margin_pct,high_discount_rate,total_orders
0,Ohio,-21.69,0.3817,469
1,Colorado,-20.33,0.2418,182
2,Tennessee,-17.42,0.2131,183
3,Illinois,-15.73,0.4634,492
4,Texas,-15.12,0.4213,985
5,North Carolina,-13.47,0.1928,249
6,Pennsylvania,-13.35,0.3969,587
7,Arizona,-9.72,0.2232,224
8,Oregon,-6.83,0.1935,124
9,Florida,-3.80,0.2193,383


**Result: r = −0.905, p < 0.0001 (n = 45 states)**

This is a very strong negative correlation — states with a higher share of heavily-discounted orders consistently show lower profit margins.

The relationship is statistically significant at p < 0.0001, meaning there is less than a 0.01% probability of observing a correlation this strong by chance alone.

**Interpretation:** Discounting is not a Central-region quirk — it is a systematic driver of margin erosion across the entire business. Central simply has the highest concentration of high-discount orders (Step 3b), which explains why its margin is the lowest.

This finding upgrades the earlier observation from correlation to a statistically validated claim: **reducing high-discount orders is the most direct lever for improving profit margins.**

### Step 3d — Ruling out product mix as a confounding variable

A natural alternative explanation is that Central sells more inherently low-margin products (e.g. Furniture), and the discount–margin correlation is simply picking up a product mix effect rather than a causal discount effect.

To test this, we repeat the state-level correlation **within each Category separately**. If the negative relationship between high discount rate and profit margin holds inside each category, then product mix cannot explain it away.

In [85]:
query = """
    SELECT
        Category,
        State,
        ROUND(SUM(Profit) / SUM(Sales) * 100, 2) AS profit_margin_pct,
        ROUND(
            CAST(COUNT(CASE WHEN Discount > 0.2 THEN 1 END) AS REAL) / COUNT(*),
        4) AS high_discount_rate,
        COUNT(*) AS total_orders
    FROM orders
    GROUP BY Category, State
    HAVING total_orders >= 5
"""

cat_state_df = pd.read_sql(query, conn)

print(f"{'Category':<20} {'r':>8} {'p-value':>10} {'n':>6}")
print("-" * 48)
for category, group in cat_state_df.groupby("Category"):
    r, p = stats.pearsonr(group["high_discount_rate"], group["profit_margin_pct"])
    print(f"{category:<20} {r:>8.3f} {p:>10.4f} {len(group):>6}")

Category                    r    p-value      n
------------------------------------------------
Furniture              -0.757     0.0000     37
Office Supplies        -0.919     0.0000     46
Technology             -0.669     0.0000     39


**Result:**

| Category | r | p-value | n |
|---|---|---|---|
| Furniture | −0.757 | < 0.0001 | 37 |
| Office Supplies | −0.919 | < 0.0001 | 46 |
| Technology | −0.669 | < 0.0001 | 39 |

All three categories show a strong, statistically significant negative correlation between high discount rate and profit margin. The relationship holds regardless of what product is being sold.

**Interpretation:** Product mix is not the confounding variable. A state with a high share of heavily-discounted orders loses margin in Furniture, Office Supplies, and Technology alike. This rules out the explanation that Central underperforms simply because it sells more low-margin product types.

**Remaining limitation:** Other confounders — regional competition intensity, customer segment mix, logistics costs — are not controlled for. This analysis establishes a strong and consistent association across categories, but stops short of a causal claim.

## Step 4 — Business Question 3: Which sub-categories are losing money in Central?

We now zoom into Central and rank sub-categories by total profit to identify where losses are concentrated.

In [86]:
query = """
    SELECT
        Category,
        `Sub-Category`,
        ROUND(AVG(Discount) * 100, 1) AS avg_discount_pct,
        ROUND(SUM(Sales), 0)          AS total_sales,
        ROUND(SUM(Profit), 0)         AS total_profit,
        COUNT(*)                       AS total_orders
    FROM orders
    WHERE Region = 'Central'
    GROUP BY Category, `Sub-Category`
    ORDER BY total_profit ASC
"""
central_df = pd.read_sql(query, conn)
central_df

,Category,Sub-Category,avg_discount_pct,total_sales,total_profit,total_orders
0,Furniture,Furnishings,40.4,15254.0,-3906.0,205
1,Furniture,Tables,26.2,39155.0,-3560.0,72
2,Office Supplies,Appliances,44.9,23582.0,-2639.0,123
3,Furniture,Bookcases,23.3,24157.0,-1998.0,50
4,Technology,Machines,32.9,26797.0,-1486.0,21
5,Office Supplies,Binders,50.9,56923.0,-1044.0,366
6,Office Supplies,Supplies,14.4,9467.0,-662.0,36
7,Office Supplies,Fasteners,13.5,778.0,237.0,55
8,Office Supplies,Labels,11.3,2451.0,1073.0,76
9,Office Supplies,Art,12.3,5765.0,1195.0,176


**Finding:** 7 out of 17 sub-categories in Central are unprofitable, accounting for a combined loss of −$15,294. The worst offenders — Furnishings (−$3,906), Tables (−$3,560), and Appliances (−$2,639) — all carry average discounts well above the 20% breakeven threshold identified in Step 3a.

The pattern is clear: the higher the average discount, the deeper the loss. But this raises a key question: **is Central uniquely over-discounting these products, or is the same behaviour happening in other regions too?** The answer determines whether the fix is a regional policy issue or a company-wide pricing strategy problem.

## Step 5 — Is Central's over-discounting unique?

We compare the average discount for the 7 loss-making sub-categories across all four regions.

- If Central's discount levels are significantly higher than other regions → the problem is a **regional management issue** (Central's sales team is granting excessive discounts)
- If all regions discount at similar levels → the problem is a **company-wide pricing strategy issue** (the product lines themselves are structurally unprofitable when discounted)

In [87]:
loss_subcats = ('Furnishings', 'Tables', 'Appliances', 'Bookcases', 'Machines', 'Binders', 'Supplies')

query = f"""
    SELECT
        `Sub-Category`,
        Region,
        ROUND(AVG(Discount) * 100, 1) AS avg_discount_pct,
        ROUND(SUM(Profit), 0)          AS total_profit
    FROM orders
    WHERE `Sub-Category` IN {loss_subcats}
    GROUP BY `Sub-Category`, Region
"""

cross_df = pd.read_sql(query, conn)

# Pivot: rows = sub-category, columns = region, values = avg_discount_pct
discount_pivot = cross_df.pivot(index='Sub-Category', columns='Region', values='avg_discount_pct')
discount_pivot.columns.name = None
discount_pivot['Central vs others'] = (
    discount_pivot['Central'] - discount_pivot[['East', 'South', 'West']].mean(axis=1)
).round(1)

print("Average Discount % by Sub-Category and Region  (Central vs others = Central minus avg of other 3)\n")
print(discount_pivot.to_string())

Average Discount % by Sub-Category and Region  (Central vs others = Central minus avg of other 3)

              Central  East  South  West  Central vs others
Sub-Category                                               
Appliances       44.9   7.3   11.1   3.1               37.7
Binders          50.9  35.3   37.6  28.2               17.2
Bookcases        23.3  22.0   10.0  22.9                5.0
Furnishings      40.4   7.8   10.7   3.3               33.1
Machines         32.9  28.4   33.3  30.3                2.2
Supplies         14.4   6.8   10.3   3.8                7.4
Tables           26.2  37.4   22.3  20.0               -0.4


**The results reveal two distinct problems:**

**Group A — Central-specific over-discounting (regional management problem)**

| Sub-Category | Central | Other regions avg | Gap |
|---|---|---|---|
| Appliances | 44.9% | 7.2% | +37.7% |
| Furnishings | 40.4% | 7.3% | +33.1% |
| Binders | 50.9% | 33.7% | +17.2% |

Central's sales team is granting dramatically higher discounts on these products than any other region. The other regions sell the same products at low discount and remain profitable. This is a **Central-specific discount policy issue**.

**Group B — Company-wide structural problem**

| Sub-Category | Central | Other regions avg | Gap |
|---|---|---|---|
| Tables | 26.2% | 26.6% | −0.4% |
| Machines | 32.9% | 30.7% | +2.2% |
| Bookcases | 23.3% | 18.3% | +5.0% |

All four regions discount these products at similar levels — and all four regions likely lose money on them. Reducing Central's discount alone would not fix this. The issue here is **company-wide product pricing**: these items may be structurally unprofitable at the discount levels currently offered across the business.

---

**Conclusion:** The two problems require two different actions:
1. **For Central management** — immediately review and cap discount authorisation for Appliances and Furnishings, where Central gives 5× the discount of other regions for no apparent reason.
2. **For company leadership** — review the cost structure and list pricing for Tables, Machines, and Bookcases, which lose money at the standard discount levels applied across all regions.